# 03 — Product Recommendation Agent Demo

This notebook demonstrates the **Product Recommendation Agent** using a ReAct loop.

It covers:
1. Basic product search
2. Multi-turn conversation (memory)
3. Ambiguous query → clarification
4. Twitter flag toggle comparison
5. Category-specific search
6. Edge case — no results

## Setup

In [1]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")

Project root: d:\IISc Group Project\IISCCapstoneProjectGroup16


In [2]:
from src.agents.product_agent import ProductRecommendationAgent

## Scenario 1: Basic Product Search

A straightforward product query to test SQL generation and retrieval.

In [ ]:
agent = ProductRecommendationAgent(
    session_id="demo-1",
    use_twitter_samples=False,
    debug=True
)

response = agent.chat("Show me the top 5 televisions under $2000 with the best ratings")
print(response)

\n==================== DEBUG: Agent Execution Started ====================
================================ Human Message =================================

Show me the top 5 laptops under $500 with the best ratings
================================== Ai Message ==================================
Tool Calls:
  query_products (fc_fd22641b-34a4-45d2-acd1-a5d2ccd12af6)
 Call ID: fc_fd22641b-34a4-45d2-acd1-a5d2ccd12af6
  Args:
    sql_query: SELECT product_id, title, price, average_rating, rating_count, features FROM product_catalog WHERE price <= 500 AND (LOWER(sub_categories) LIKE '%laptop%' OR LOWER(main_category) LIKE '%laptop%') ORDER BY average_rating DESC, rating_count DESC LIMIT 5;
================================= Tool Message =================================
Name: query_products

[
  {
    "product_id": "B07M95WPYV",
    "title": "Yoga Frog Sticker",
    "price": 3.19,
    "average_rating": 5.0,
    "rating_count": 34,
    "features": "[]"
  },
  {
    "product_id": "B0B9TBMQ94",

## Scenario 2: Multi-Turn Conversation (Memory)

Follow-up query that relies on conversation history.

In [ ]:
# Continuing from Scenario 1 — same agent instance
response = agent.chat("Which of those has the most reviews?")
print(response)

## Scenario 3: Ambiguous Query → Clarification

A vague query that should trigger the agent to ask for more details.

In [ ]:
agent2 = ProductRecommendationAgent(
    session_id="demo-2",
    use_twitter_samples=False,
)

response = agent2.chat("I need something for my home")
print(response)

## Scenario 4: Twitter Flag Toggle

Compare the prompt and behavior with `use_twitter_samples=True` vs `False`.

> **Note**: Since twitter retrieval is a placeholder, the main difference
> is in the prompt structure (tool availability and priority instructions).

In [ ]:
# With twitter samples enabled
agent_twitter = ProductRecommendationAgent(
    session_id="demo-3a",
    use_twitter_samples=True,
)

response_with_twitter = agent_twitter.chat("What are good wireless earbuds?")
print("=== With Twitter Samples ===")
print(response_with_twitter)

In [ ]:
# Without twitter samples
agent_no_twitter = ProductRecommendationAgent(
    session_id="demo-3b",
    use_twitter_samples=False,
)

response_no_twitter = agent_no_twitter.chat("What are good wireless earbuds?")
print("=== Without Twitter Samples ===")
print(response_no_twitter)

## Scenario 5: Category-Specific Search

Tests the agent's ability to filter by `main_category` or `sub_categories`.

In [ ]:
agent3 = ProductRecommendationAgent(
    session_id="demo-4",
    use_twitter_samples=False,
)

response = agent3.chat("What are the best rated electronics accessories?")
print(response)

## Scenario 6: Edge Case — No Results

Query for a product type that likely doesn't exist in the catalog.

In [ ]:
agent4 = ProductRecommendationAgent(
    session_id="demo-5",
    use_twitter_samples=False,
)

response = agent4.chat("Do you have any quantum computers for sale?")
print(response)

---

## Summary

| Scenario | What was tested |
|---|---|
| 1 | SQL query generation, product retrieval, formatting |
| 2 | Conversation memory for follow-up queries |
| 3 | Clarification tool for ambiguous requests |
| 4 | Twitter flag toggles prompt structure |
| 5 | Category filtering via sub_categories |
| 6 | Graceful handling of empty results |